# LightGBM Strict-CV Screen

This notebook tests LightGBM under the stricter fold assignment created after the analog-gate public miss. The goal is not to package a Kaggle submission yet; it is to see whether boosted trees can learn useful nonlinear interactions under `strict_hybrid_fold` without relying on a too-optimistic local split.


## Modeling Design

We predict target-zone signed delta from each well's last known TVT, then add the last known TVT back for RMSE. Features are built only from row geometry, boundary distances, known-zone summaries, GR, typewell/alignment descriptors, and strict fold metadata. Rows are sampled deterministically per well to keep the first screen fast and repeatable.


In [1]:
from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

ROOT = Path.cwd()
if not (ROOT / "artifacts").exists():
    ROOT = ROOT.parent.parent.parent

RUN_ID = "2026-06-26__lightgbm_strict_cv"
RUN_DIR = ROOT / "artifacts" / "logs" / "runs" / RUN_ID
TABLE_DIR = ROOT / "artifacts" / "tables"
REPORT_DIR = ROOT / "artifacts" / "reports"
for path in [RUN_DIR, TABLE_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "fold_column": "strict_hybrid_fold",
    "stress_fold_column": "analog_distance_fold",
    "max_target_rows_per_well": 160,
    "random_state": 20260626,
    "current_public_baseline": 14.514,
    "strict_cv_promotion_gate": 0.10,
}
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

"Set up paths and deterministic screen configuration."
print(ROOT)
print(RUN_DIR)


C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\logs\runs\2026-06-26__lightgbm_strict_cv


## Feature Construction


In [2]:
folds = pd.read_csv(TABLE_DIR / "validation__strict_cv_fold_assignments__2026-06-26.csv")
descriptors = pd.read_csv(TABLE_DIR / "model__typewell_alignment_feature_descriptors__2026-06-23.csv")
well_meta = folds.merge(descriptors, on="wellname", how="left")
well_meta = well_meta.drop_duplicates(subset=["wellname"]).reset_index(drop=True)

formation_cols = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
descriptor_cols = [
    "known_rows", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300",
    "xy_span_known", "z_drop_known", "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr",
    "typewell_best_lag_tvt", "typewell_best_lag_mae", "selected_analog_distance", "extrapolation_stress_score",
    "analog_stress_score", "combined_stress_score", "descriptor_cluster",
]

def safe_divide(num, den):
    return num / (den + 1e-9)

def build_target_rows(csv_path, meta_row):
    df = pd.read_csv(csv_path)
    df = df.sort_values("MD").reset_index(drop=True)
    known_mask = df["TVT_input"].notna()
    if known_mask.sum() == 0 or (~known_mask).sum() == 0:
        return pd.DataFrame()
    last_known_idx = int(np.flatnonzero(known_mask.to_numpy())[-1])
    known = df.loc[:last_known_idx].copy()
    target = df.loc[last_known_idx + 1:].copy()
    if target.empty:
        return pd.DataFrame()
    if len(target) > CONFIG["max_target_rows_per_well"]:
        sample_idx = np.linspace(0, len(target) - 1, CONFIG["max_target_rows_per_well"]).round().astype(int)
        target = target.iloc[np.unique(sample_idx)].copy()
    last = known.iloc[-1]
    first = df.iloc[0]
    known_gr = known["GR"].astype(float).clip(0, 300)
    gr_fill = float(known_gr.tail(300).median()) if known_gr.notna().any() else 0.0
    target_gr = target["GR"].astype(float).clip(0, 300).fillna(gr_fill)
    target = target.reset_index(drop=False).rename(columns={"index": "row_idx_original"})
    out = pd.DataFrame({
        "wellname": meta_row["wellname"],
        "row_idx_original": target["row_idx_original"].astype(float),
        "MD": target["MD"].astype(float),
        "X": target["X"].astype(float),
        "Y": target["Y"].astype(float),
        "Z": target["Z"].astype(float),
        "GR": target_gr.to_numpy(),
        "TVT": target["TVT"].astype(float),
        "last_known_tvt_feature": float(last["TVT"]),
        "last_known_md": float(last["MD"]),
        "last_known_x": float(last["X"]),
        "last_known_y": float(last["Y"]),
        "last_known_z": float(last["Z"]),
        "first_md": float(first["MD"]),
    })
    out["target_delta"] = out["TVT"] - out["last_known_tvt_feature"]
    out["md_from_last_known"] = out["MD"] - out["last_known_md"]
    out["md_from_start"] = out["MD"] - out["first_md"]
    out["eval_progress"] = safe_divide(np.arange(1, len(out) + 1), len(out))
    out["dx_from_last"] = out["X"] - out["last_known_x"]
    out["dy_from_last"] = out["Y"] - out["last_known_y"]
    out["dz_from_last"] = out["Z"] - out["last_known_z"]
    out["horizontal_from_last"] = np.sqrt(out["dx_from_last"] ** 2 + out["dy_from_last"] ** 2)
    out["distance_from_last"] = np.sqrt(out["horizontal_from_last"] ** 2 + out["dz_from_last"] ** 2)
    out["distance_log1p"] = np.log1p(out["distance_from_last"].clip(lower=0))
    for col in formation_cols:
        out[f"dist_to_{col}"] = target[col].astype(float).to_numpy() - out["Z"]
        out[f"abs_dist_to_{col}"] = np.abs(out[f"dist_to_{col}"])
    out["min_abs_formation_dist"] = out[[f"abs_dist_to_{c}" for c in formation_cols]].min(axis=1)
    out["mean_abs_formation_dist"] = out[[f"abs_dist_to_{c}" for c in formation_cols]].mean(axis=1)
    for col in [CONFIG["fold_column"], CONFIG["stress_fold_column"], "descriptor_cluster_fold", "extrapolation_fold"]:
        out[col] = int(meta_row[col])
    for col in descriptor_cols:
        out[col] = float(meta_row[col]) if pd.notna(meta_row[col]) else np.nan
    return out

rows = []
train_dir = ROOT / "data" / "raw" / "train"
for meta_row in well_meta.to_dict(orient="records"):
    csv_path = train_dir / f"{meta_row['wellname']}__horizontal_well.csv"
    if csv_path.exists():
        rows.append(build_target_rows(csv_path, meta_row))
data = pd.concat(rows, ignore_index=True)

feature_cols = [
    "row_idx_original", "MD", "X", "Y", "Z", "GR", "last_known_tvt_feature", "last_known_md",
    "md_from_last_known", "md_from_start", "eval_progress", "dx_from_last", "dy_from_last", "dz_from_last",
    "horizontal_from_last", "distance_from_last", "distance_log1p", "min_abs_formation_dist", "mean_abs_formation_dist",
    *[f"dist_to_{c}" for c in formation_cols], *[f"abs_dist_to_{c}" for c in formation_cols], *descriptor_cols,
]
feature_cols = [col for col in feature_cols if col in data.columns]

feature_preview_path = TABLE_DIR / "model__lightgbm_strict_cv_feature_preview__2026-06-26.csv"
data.head(500).to_csv(feature_preview_path, index=False)
shutil.copy2(feature_preview_path, RUN_DIR / feature_preview_path.name)

print(data.shape)
print(len(feature_cols), feature_cols[:12])
data.head()


(123680, 60)
49 ['row_idx_original', 'MD', 'X', 'Y', 'Z', 'GR', 'last_known_tvt_feature', 'last_known_md', 'md_from_last_known', 'md_from_start', 'eval_progress', 'dx_from_last']


,wellname,row_idx_original,MD,X,Y,Z,GR,TVT,last_known_tvt_feature,last_known_md,last_known_x,last_known_y,last_known_z,first_md,target_delta,md_from_last_known,md_from_start,eval_progress,dx_from_last,dy_from_last,dz_from_last,horizontal_from_last,distance_from_last,distance_log1p,dist_to_ANCC,abs_dist_to_ANCC,dist_to_ASTNU,abs_dist_to_ASTNU,dist_to_ASTNL,abs_dist_to_ASTNL,dist_to_EGFDU,abs_dist_to_EGFDU,dist_to_EGFDL,abs_dist_to_EGFDL,dist_to_BUDA,abs_dist_to_BUDA,min_abs_formation_dist,mean_abs_formation_dist,strict_hybrid_fold,analog_distance_fold,descriptor_cluster_fold,extrapolation_fold,known_rows,known_tvt_span,known_md_span,known_gr_mean_last300,known_gr_std_last300,known_gr_slope_last300,xy_span_known,z_drop_known,typewell_corr_last300,typewell_mae_last300,typewell_best_lag_corr,typewell_best_lag_tvt,typewell_best_lag_mae,selected_analog_distance,extrapolation_stress_score,analog_stress_score,combined_stress_score,descriptor_cluster
0,000d7d20,1442.0,12909.0,2983537.06,1070212.72,-9735.07,106.984057,11747.38,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,0.01,1.0,1442.0,0.00625,-0.03,1.00,0.01,1.000450,1.000500,0.693397,374.12,374.12,200.07,200.07,172.29,172.29,98.94,98.94,63.97,63.97,-76.42,76.42,63.97,164.301667,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0.398691,11.532797,0.553606,-22.5,19.894136,0.316157,0.268111,0.241915,0.234039,1.0
1,000d7d20,1466.0,12933.0,2983536.35,1070236.70,-9734.74,106.984057,11747.51,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,0.14,25.0,1466.0,0.01250,-0.74,24.98,0.34,24.990958,24.993271,3.257838,374.25,374.25,200.20,200.20,172.42,172.42,99.07,99.07,64.10,64.10,-76.29,76.29,64.10,164.388333,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0.398691,11.532797,0.553606,-22.5,19.894136,0.316157,0.268111,0.241915,0.234039,1.0
2,000d7d20,1490.0,12957.0,2983535.66,1070260.69,-9734.24,106.984057,11747.47,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,0.10,49.0,1490.0,0.01875,-1.43,48.97,0.84,48.990875,48.998075,3.911985,374.21,374.21,200.16,200.16,172.38,172.38,99.03,99.03,64.06,64.06,-76.33,76.33,64.06,164.361667,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0.398691,11.532797,0.553606,-22.5,19.894136,0.316157,0.268111,0.241915,0.234039,1.0
3,000d7d20,1514.0,12981.0,2983534.97,1070284.67,-9733.57,81.544322,11747.81,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,0.44,73.0,1514.0,0.02500,-2.12,72.95,1.51,72.980798,72.996418,4.304017,374.55,374.55,200.50,200.50,172.72,172.72,99.37,99.37,64.40,64.40,-75.99,75.99,64.40,164.588333,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0.398691,11.532797,0.553606,-22.5,19.894136,0.316157,0.268111,0.241915,0.234039,1.0
4,000d7d20,1538.0,13005.0,2983534.20,1070308.65,-9732.85,106.984057,11748.36,11747.37,12908.0,2983537.09,1070211.72,-9735.08,11467.0,0.99,97.0,1538.0,0.03125,-2.89,96.93,2.23,96.973074,96.998711,4.584954,375.11,375.11,201.06,201.06,173.28,173.28,99.93,99.93,64.96,64.96,-75.43,75.43,64.96,164.961667,2,1,1,1,1442.0,520.1,1441.0,106.326237,6.06826,3.165573,1190.815381,-476.51,-0.398691,11.532797,0.553606,-22.5,19.894136,0.316157,0.268111,0.241915,0.234039,1.0


## Candidate Grid


In [3]:
candidates = {
    "lgbm_conservative_l2": {
        "n_estimators": 1200, "learning_rate": 0.025, "num_leaves": 31, "max_depth": 8, "min_child_samples": 120,
        "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 0.05, "reg_lambda": 3.0,
    },
    "lgbm_medium_l2": {
        "n_estimators": 1200, "learning_rate": 0.03, "num_leaves": 63, "max_depth": 10, "min_child_samples": 80,
        "subsample": 0.90, "colsample_bytree": 0.90, "reg_alpha": 0.02, "reg_lambda": 2.0,
    },
    "lgbm_shallow_robust": {
        "n_estimators": 900, "learning_rate": 0.035, "num_leaves": 23, "max_depth": 6, "min_child_samples": 160,
        "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.10, "reg_lambda": 5.0,
    },
}
base_params = {
    "objective": "huber",
    "random_state": CONFIG["random_state"],
    "n_jobs": -1,
    "verbosity": -1,
}
print(pd.DataFrame(candidates).T)


                      n_estimators  learning_rate  num_leaves  max_depth  min_child_samples  subsample  colsample_bytree  reg_alpha  reg_lambda
lgbm_conservative_l2        1200.0          0.025        31.0        8.0              120.0       0.85              0.85       0.05         3.0
lgbm_medium_l2              1200.0          0.030        63.0       10.0               80.0       0.90              0.90       0.02         2.0
lgbm_shallow_robust          900.0          0.035        23.0        6.0              160.0       0.80              0.80       0.10         5.0


## Strict CV Training Loop


In [4]:
def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)

X_all = data[feature_cols].replace([np.inf, -np.inf], np.nan)
y_delta = data["target_delta"].astype(float)
y_tvt = data["TVT"].astype(float)
baseline_pred = data["last_known_tvt_feature"].astype(float)

fold_metrics = []
pred_parts = []
importance_parts = []
start_time = time.time()

for candidate, params in candidates.items():
    candidate_start = time.time()
    for fold in sorted(data[CONFIG["fold_column"]].unique()):
        valid_mask = data[CONFIG["fold_column"]].eq(fold)
        train_mask = ~valid_mask
        model = LGBMRegressor(**base_params, **params)
        model.fit(
            X_all.loc[train_mask], y_delta.loc[train_mask],
            eval_set=[(X_all.loc[valid_mask], y_delta.loc[valid_mask])],
            eval_metric="rmse",
            callbacks=[early_stopping(80, verbose=False), log_evaluation(period=0)],
        )
        pred_delta = model.predict(X_all.loc[valid_mask], num_iteration=model.best_iteration_)
        pred_tvt = baseline_pred.loc[valid_mask].to_numpy() + pred_delta
        base = baseline_pred.loc[valid_mask].to_numpy()
        actual = y_tvt.loc[valid_mask].to_numpy()
        fold_metrics.append({
            "candidate": candidate,
            "fold": int(fold),
            "fold_column": CONFIG["fold_column"],
            "n_train_rows": int(train_mask.sum()),
            "n_valid_rows": int(valid_mask.sum()),
            "best_iteration": int(model.best_iteration_ or params["n_estimators"]),
            "rmse": rmse(actual, pred_tvt),
            "mae": float(mean_absolute_error(actual, pred_tvt)),
            "r2": float(r2_score(actual, pred_tvt)),
            "last_known_hold_rmse": rmse(actual, base),
            "train_time_s": float(time.time() - candidate_start),
        })
        pred_tmp = data.loc[valid_mask, ["wellname", CONFIG["fold_column"], CONFIG["stress_fold_column"], "eval_progress", "selected_analog_distance", "combined_stress_score"]].copy()
        pred_tmp["candidate"] = candidate
        pred_tmp["actual_tvt"] = actual
        pred_tmp["pred_tvt"] = pred_tvt
        pred_tmp["baseline_tvt"] = base
        pred_tmp["residual"] = actual - pred_tvt
        pred_tmp["baseline_residual"] = actual - base
        pred_parts.append(pred_tmp)
        importance_parts.append(pd.DataFrame({"candidate": candidate, "fold": int(fold), "feature": feature_cols, "importance": model.feature_importances_}))

fold_metrics = pd.DataFrame(fold_metrics)
predictions = pd.concat(pred_parts, ignore_index=True)
importance = pd.concat(importance_parts, ignore_index=True)
print(f"elapsed_s={time.time() - start_time:.1f}")
fold_metrics


elapsed_s=132.3


,candidate,fold,fold_column,n_train_rows,n_valid_rows,best_iteration,rmse,mae,r2,last_known_hold_rmse,train_time_s
0,lgbm_conservative_l2,1,strict_hybrid_fold,81920,41760,1200,14.587262,9.960881,0.999388,16.419286,12.135101
1,lgbm_conservative_l2,2,strict_hybrid_fold,82560,41120,1200,12.846927,9.297691,0.999591,14.948338,25.386202
2,lgbm_conservative_l2,3,strict_hybrid_fold,82880,40800,1200,13.386485,9.684797,0.999548,15.352430,41.136429
3,lgbm_medium_l2,1,strict_hybrid_fold,81920,41760,1200,14.153451,9.637983,0.999424,16.419286,20.139019
4,lgbm_medium_l2,2,strict_hybrid_fold,82560,41120,1200,12.564543,9.132537,0.999609,14.948338,45.497382
5,lgbm_medium_l2,3,strict_hybrid_fold,82880,40800,1200,12.990907,9.390969,0.999574,15.352430,68.405415
6,lgbm_shallow_robust,1,strict_hybrid_fold,81920,41760,900,14.742171,10.034260,0.999375,16.419286,6.880444
7,lgbm_shallow_robust,2,strict_hybrid_fold,82560,41120,900,13.053635,9.458882,0.999578,14.948338,14.196290
8,lgbm_shallow_robust,3,strict_hybrid_fold,82880,40800,900,13.386468,9.695180,0.999548,15.352430,22.728527


## Diagnostics and Exports


In [5]:
summary = (
    fold_metrics.groupby("candidate")
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        r2_mean=("r2", "mean"),
        last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        best_iteration_mean=("best_iteration", "mean"),
        n_valid_rows=("n_valid_rows", "sum"),
    )
    .reset_index()
)
summary["rmse_delta_vs_last_known_hold"] = summary["rmse_mean"] - summary["last_known_hold_rmse_mean"]
summary["strict_cv_margin_vs_public_baseline"] = summary["rmse_mean"] - CONFIG["current_public_baseline"]
summary["promote_to_stress_test"] = summary["strict_cv_margin_vs_public_baseline"] <= -CONFIG["strict_cv_promotion_gate"]
summary = summary.sort_values("rmse_mean").reset_index(drop=True)
best_candidate = str(summary.iloc[0]["candidate"])

predictions["progress_zone"] = pd.cut(predictions["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])
predictions["analog_distance_bin"] = pd.qcut(predictions["selected_analog_distance"], q=5, labels=["closest", "near", "middle", "far", "farthest"], duplicates="drop")

zone_summary = (
    predictions.groupby(["candidate", "progress_zone"], observed=True)
    .apply(lambda g: pd.Series({
        "n_rows": len(g),
        "rmse": rmse(g["actual_tvt"], g["pred_tvt"]),
        "baseline_rmse": rmse(g["actual_tvt"], g["baseline_tvt"]),
        "mean_residual": float(g["residual"].mean()),
    }), include_groups=False)
    .reset_index()
)
zone_summary["rmse_delta_vs_hold"] = zone_summary["rmse"] - zone_summary["baseline_rmse"]

analog_bin_summary = (
    predictions.groupby(["candidate", "analog_distance_bin"], observed=True)
    .apply(lambda g: pd.Series({
        "n_rows": len(g),
        "n_wells": g["wellname"].nunique(),
        "mean_selected_analog_distance": float(g["selected_analog_distance"].mean()),
        "rmse": rmse(g["actual_tvt"], g["pred_tvt"]),
        "baseline_rmse": rmse(g["actual_tvt"], g["baseline_tvt"]),
    }), include_groups=False)
    .reset_index()
)
analog_bin_summary["rmse_delta_vs_hold"] = analog_bin_summary["rmse"] - analog_bin_summary["baseline_rmse"]

well_diag = (
    predictions.loc[predictions["candidate"].eq(best_candidate)]
    .groupby("wellname")
    .agg(
        n_rows=("actual_tvt", "size"),
        fold=(CONFIG["fold_column"], "first"),
        stress_fold=(CONFIG["stress_fold_column"], "first"),
        mean_selected_analog_distance=("selected_analog_distance", "mean"),
        mean_combined_stress=("combined_stress_score", "mean"),
        mean_residual=("residual", "mean"),
        mean_abs_residual=("residual", lambda s: float(np.mean(np.abs(s)))),
        rmse=("residual", lambda s: float(np.sqrt(np.mean(np.square(s))))),
    )
    .reset_index()
    .sort_values("rmse", ascending=False)
)

importance_summary = (
    importance.groupby(["candidate", "feature"])["importance"].mean().reset_index()
    .sort_values(["candidate", "importance"], ascending=[True, False])
)

paths = {
    "summary": TABLE_DIR / "model__lightgbm_strict_cv_summary__2026-06-26.csv",
    "fold_metrics": TABLE_DIR / "model__lightgbm_strict_cv_fold_metrics__2026-06-26.csv",
    "zone_summary": TABLE_DIR / "model__lightgbm_strict_cv_zone_summary__2026-06-26.csv",
    "analog_bin_summary": TABLE_DIR / "model__lightgbm_strict_cv_analog_bin_summary__2026-06-26.csv",
    "well_diagnostics": TABLE_DIR / "model__lightgbm_strict_cv_well_diagnostics__2026-06-26.csv",
    "feature_importance": TABLE_DIR / "model__lightgbm_strict_cv_feature_importance__2026-06-26.csv",
}
exports = {
    "summary": summary, "fold_metrics": fold_metrics, "zone_summary": zone_summary, "analog_bin_summary": analog_bin_summary,
    "well_diagnostics": well_diag, "feature_importance": importance_summary,
}
for key, df in exports.items():
    df.to_csv(paths[key], index=False)
    shutil.copy2(paths[key], RUN_DIR / paths[key].name)

print(summary.to_string(index=False))
importance_summary.groupby("candidate").head(10)


           candidate  rmse_mean  rmse_std  mae_mean  r2_mean  last_known_hold_rmse_mean  best_iteration_mean  n_valid_rows  rmse_delta_vs_last_known_hold  strict_cv_margin_vs_public_baseline  promote_to_stress_test
      lgbm_medium_l2  13.236300  0.822387  9.387163 0.999536                  15.573352               1200.0        123680                      -2.337052                            -1.277700                    True
lgbm_conservative_l2  13.606891  0.890857  9.647790 0.999509                  15.573352               1200.0        123680                      -1.966460                            -0.907109                    True
 lgbm_shallow_robust  13.727425  0.894414  9.729441 0.999500                  15.573352                900.0        123680                      -1.845927                            -0.786575                    True


,candidate,feature,importance
26,lgbm_conservative_l2,extrapolation_stress_score,3167.000000
35,lgbm_conservative_l2,last_known_tvt_feature,1683.666667
30,lgbm_conservative_l2,known_gr_std_last300,1320.333333
33,lgbm_conservative_l2,known_tvt_span,1272.000000
43,lgbm_conservative_l2,typewell_best_lag_mae,1256.000000
2,lgbm_conservative_l2,X,1234.666667
17,lgbm_conservative_l2,dist_to_BUDA,1218.333333
39,lgbm_conservative_l2,min_abs_formation_dist,1216.666667
28,lgbm_conservative_l2,known_gr_mean_last300,1208.333333
12,lgbm_conservative_l2,combined_stress_score,1205.666667


## Report and Decision


In [6]:
best = summary.iloc[0]
promote = bool(best["promote_to_stress_test"])
decision = "promote to analog-distance stress test" if promote else "do not promote yet"
findings = pd.DataFrame([
    {"finding": "Best LightGBM strict-CV candidate", "value": f"{best['candidate']} RMSE {best['rmse_mean']:.4f}", "implication": decision},
    {"finding": "Delta vs last-known hold", "value": f"{best['rmse_delta_vs_last_known_hold']:.4f}", "implication": "Negative means LightGBM beats the simple hold baseline on the sampled strict folds."},
    {"finding": "Margin vs current public baseline reference", "value": f"{best['strict_cv_margin_vs_public_baseline']:.4f}", "implication": "This is a rough reference only because public LB and strict sampled CV are not directly comparable."},
    {"finding": "Best feature family", "value": str(importance_summary.loc[importance_summary['candidate'].eq(best['candidate'])].iloc[0]['feature']), "implication": "Inspect whether the model leans on robust geometry or fragile identifiers."},
])
findings_path = TABLE_DIR / "model__lightgbm_strict_cv_findings__2026-06-26.csv"
findings.to_csv(findings_path, index=False)
shutil.copy2(findings_path, RUN_DIR / findings_path.name)

comparison_row = pd.DataFrame([{
    "run_id": RUN_ID,
    "model_family": "tree",
    "model_name": str(best["candidate"]),
    "cv_rmse_mean": float(best["rmse_mean"]),
    "cv_rmse_std": float(best["rmse_std"]),
    "train_time_s": float(fold_metrics["train_time_s"].max()),
    "n_features": int(len(feature_cols)),
    "notes": f"LightGBM strict CV on {CONFIG['fold_column']}; {decision}",
}])
comparison_path = TABLE_DIR / "model_comparison.csv"
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    comparison = comparison.loc[comparison["run_id"].ne(RUN_ID)]
    comparison = pd.concat([comparison, comparison_row], ignore_index=True)
else:
    comparison = comparison_row
comparison.to_csv(comparison_path, index=False)
shutil.copy2(comparison_path, RUN_DIR / "model_comparison.csv")

report = f"""# LightGBM Strict-CV Screen - 2026-06-26

## Summary

Best candidate: `{best['candidate']}` with strict sampled CV RMSE **{best['rmse_mean']:.4f}** (std **{best['rmse_std']:.4f}**) over `{CONFIG['fold_column']}`. The same sampled rows have last-known hold RMSE **{best['last_known_hold_rmse_mean']:.4f}**, so the LightGBM delta vs hold is **{best['rmse_delta_vs_last_known_hold']:.4f}**.

## Decision

Decision: **{decision}**. The promotion gate for a second stress test is at least **{CONFIG['strict_cv_promotion_gate']:.2f} RMSE** better than the current public-baseline reference, while remembering that public LB and sampled strict CV are not directly comparable.

## Notes

- Target is signed TVT delta from the last known TVT, with predictions transformed back to TVT for RMSE.
- Rows are deterministically sampled at up to **{CONFIG['max_target_rows_per_well']}** target rows per well.
- Next step is to run the same best candidate under `analog_distance_fold` only if this result is convincing enough.
"""
report_path = REPORT_DIR / "model__lightgbm_strict_cv__2026-06-26.md"
report_path.write_text(report, encoding="utf-8")
shutil.copy2(report_path, RUN_DIR / report_path.name)

print(report)


# LightGBM Strict-CV Screen - 2026-06-26

## Summary

Best candidate: `lgbm_medium_l2` with strict sampled CV RMSE **13.2363** (std **0.8224**) over `strict_hybrid_fold`. The same sampled rows have last-known hold RMSE **15.5734**, so the LightGBM delta vs hold is **-2.3371**.

## Decision

Decision: **promote to analog-distance stress test**. The promotion gate for a second stress test is at least **0.10 RMSE** better than the current public-baseline reference, while remembering that public LB and sampled strict CV are not directly comparable.

## Notes

- Target is signed TVT delta from the last known TVT, with predictions transformed back to TVT for RMSE.
- Rows are deterministically sampled at up to **160** target rows per well.
- Next step is to run the same best candidate under `analog_distance_fold` only if this result is convincing enough.



## Analog-Distance Stress Test


In [7]:
stress_candidate = best_candidate
stress_params = candidates[stress_candidate]
stress_fold_metrics = []
stress_pred_parts = []
stress_start = time.time()

for fold in sorted(data[CONFIG["stress_fold_column"]].unique()):
    valid_mask = data[CONFIG["stress_fold_column"]].eq(fold)
    train_mask = ~valid_mask
    model = LGBMRegressor(**base_params, **stress_params)
    model.fit(
        X_all.loc[train_mask], y_delta.loc[train_mask],
        eval_set=[(X_all.loc[valid_mask], y_delta.loc[valid_mask])],
        eval_metric="rmse",
        callbacks=[early_stopping(80, verbose=False), log_evaluation(period=0)],
    )
    pred_delta = model.predict(X_all.loc[valid_mask], num_iteration=model.best_iteration_)
    pred_tvt = baseline_pred.loc[valid_mask].to_numpy() + pred_delta
    base = baseline_pred.loc[valid_mask].to_numpy()
    actual = y_tvt.loc[valid_mask].to_numpy()
    stress_fold_metrics.append({
        "candidate": stress_candidate,
        "fold": int(fold),
        "fold_column": CONFIG["stress_fold_column"],
        "n_train_rows": int(train_mask.sum()),
        "n_valid_rows": int(valid_mask.sum()),
        "best_iteration": int(model.best_iteration_ or stress_params["n_estimators"]),
        "rmse": rmse(actual, pred_tvt),
        "mae": float(mean_absolute_error(actual, pred_tvt)),
        "r2": float(r2_score(actual, pred_tvt)),
        "last_known_hold_rmse": rmse(actual, base),
        "train_time_s": float(time.time() - stress_start),
    })
    pred_tmp = data.loc[valid_mask, ["wellname", CONFIG["fold_column"], CONFIG["stress_fold_column"], "eval_progress", "selected_analog_distance", "combined_stress_score"]].copy()
    pred_tmp["candidate"] = stress_candidate
    pred_tmp["stress_fold"] = int(fold)
    pred_tmp["actual_tvt"] = actual
    pred_tmp["pred_tvt"] = pred_tvt
    pred_tmp["baseline_tvt"] = base
    pred_tmp["residual"] = actual - pred_tvt
    stress_pred_parts.append(pred_tmp)

stress_fold_metrics = pd.DataFrame(stress_fold_metrics)
stress_predictions = pd.concat(stress_pred_parts, ignore_index=True)
stress_summary = (
    stress_fold_metrics.groupby(["candidate", "fold_column"])
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        best_iteration_mean=("best_iteration", "mean"),
        n_valid_rows=("n_valid_rows", "sum"),
    )
    .reset_index()
)
stress_summary["rmse_delta_vs_last_known_hold"] = stress_summary["rmse_mean"] - stress_summary["last_known_hold_rmse_mean"]
stress_summary["strict_cv_margin_vs_public_baseline"] = stress_summary["rmse_mean"] - CONFIG["current_public_baseline"]

stress_fold_path = TABLE_DIR / "model__lightgbm_strict_cv_analog_distance_fold_metrics__2026-06-26.csv"
stress_summary_path = TABLE_DIR / "model__lightgbm_strict_cv_analog_distance_summary__2026-06-26.csv"
stress_fold_metrics.to_csv(stress_fold_path, index=False)
stress_summary.to_csv(stress_summary_path, index=False)
shutil.copy2(stress_fold_path, RUN_DIR / stress_fold_path.name)
shutil.copy2(stress_summary_path, RUN_DIR / stress_summary_path.name)

print(stress_summary.to_string(index=False))
stress_fold_metrics


     candidate          fold_column  rmse_mean  rmse_std  mae_mean  last_known_hold_rmse_mean  best_iteration_mean  n_valid_rows  rmse_delta_vs_last_known_hold  strict_cv_margin_vs_public_baseline
lgbm_medium_l2 analog_distance_fold  13.445268  0.833963  9.625008                  15.583305          1158.666667        123680                      -2.138037                            -1.068732


,candidate,fold,fold_column,n_train_rows,n_valid_rows,best_iteration,rmse,mae,r2,last_known_hold_rmse,train_time_s
0,lgbm_medium_l2,1,analog_distance_fold,82400,41280,1077,12.848265,9.509022,0.999551,14.993008,18.490010
1,lgbm_medium_l2,2,analog_distance_fold,82560,41120,1200,13.089412,9.581249,0.999554,15.585567,41.857633
2,lgbm_medium_l2,3,analog_distance_fold,82400,41280,1199,14.398127,9.784753,0.999481,16.171340,62.676995


## Final Decision With Stress Test


In [8]:
stress_best = stress_summary.iloc[0]
stress_margin = float(stress_best["strict_cv_margin_vs_public_baseline"])
final_decision = "prepare Kaggle-safe LightGBM package after one reproducibility/tuning pass" if stress_margin <= -CONFIG["strict_cv_promotion_gate"] else "continue local tuning before Kaggle"

findings2 = pd.DataFrame([
    {"finding": "Best strict-hybrid candidate", "value": f"{best['candidate']} RMSE {best['rmse_mean']:.4f}", "implication": "Promising nonlinear feature learner under default strict folds."},
    {"finding": "Analog-distance stress RMSE", "value": f"{stress_best['rmse_mean']:.4f}", "implication": final_decision},
    {"finding": "Stress delta vs last-known hold", "value": f"{stress_best['rmse_delta_vs_last_known_hold']:.4f}", "implication": "Checks whether LightGBM survives the weak-analog fold design."},
])
findings_path = TABLE_DIR / "model__lightgbm_strict_cv_findings__2026-06-26.csv"
combined_findings = pd.concat([findings, findings2], ignore_index=True).drop_duplicates(subset=["finding"], keep="last")
combined_findings.to_csv(findings_path, index=False)
shutil.copy2(findings_path, RUN_DIR / findings_path.name)

report = f"""# LightGBM Strict-CV Screen - 2026-06-26

## Summary

Best default strict-CV candidate: `{best['candidate']}` with `strict_hybrid_fold` RMSE **{best['rmse_mean']:.4f}** (std **{best['rmse_std']:.4f}**). The sampled last-known hold baseline is **{best['last_known_hold_rmse_mean']:.4f}**, so the LightGBM delta vs hold is **{best['rmse_delta_vs_last_known_hold']:.4f}**.

## Analog-Distance Stress Test

The same `{stress_candidate}` configuration scored **{stress_best['rmse_mean']:.4f}** RMSE on `analog_distance_fold`, with last-known hold RMSE **{stress_best['last_known_hold_rmse_mean']:.4f}** and delta vs hold **{stress_best['rmse_delta_vs_last_known_hold']:.4f}**.

## Decision

Decision: **{final_decision}**. The result is strong enough to keep pursuing boosted trees, but Kaggle packaging should wait until one reproducibility/tuning pass confirms this was not created by a sampled-row artifact.

## Notes

- Target is signed TVT delta from the last known TVT, transformed back to TVT for RMSE.
- Rows are deterministically sampled at up to **{CONFIG['max_target_rows_per_well']}** target rows per well.
- The strongest features include extrapolation stress, last-known TVT, formation distance, GR summaries, typewell errors, and selected analog distance.
"""
report_path = REPORT_DIR / "model__lightgbm_strict_cv__2026-06-26.md"
report_path.write_text(report, encoding="utf-8")
shutil.copy2(report_path, RUN_DIR / report_path.name)
print(report)


# LightGBM Strict-CV Screen - 2026-06-26

## Summary

Best default strict-CV candidate: `lgbm_medium_l2` with `strict_hybrid_fold` RMSE **13.2363** (std **0.8224**). The sampled last-known hold baseline is **15.5734**, so the LightGBM delta vs hold is **-2.3371**.

## Analog-Distance Stress Test

The same `lgbm_medium_l2` configuration scored **13.4453** RMSE on `analog_distance_fold`, with last-known hold RMSE **15.5833** and delta vs hold **-2.1380**.

## Decision

Decision: **prepare Kaggle-safe LightGBM package after one reproducibility/tuning pass**. The result is strong enough to keep pursuing boosted trees, but Kaggle packaging should wait until one reproducibility/tuning pass confirms this was not created by a sampled-row artifact.

## Notes

- Target is signed TVT delta from the last known TVT, transformed back to TVT for RMSE.
- Rows are deterministically sampled at up to **160** target rows per well.
- The strongest features include extrapolation stress, last-known TVT, form